# NR-PANNs — huấn luyện ESC-50 bằng Jupyter Notebook

Notebook này **khởi tạo lại lần train ESC-50** từ checkpoint AudioSet-pretrained `Wavegram_Logmel_Cnn14`. NRE-V3, RGHF và classifier khởi tạo mới, trừ khi chỉ định checkpoint NRE. Tất cả trọng số PANNs **tham gia forward** đều được fine-tune.

**Cấu trúc yêu cầu** (đường dẫn `ROOT` sửa ở cell 3):
```text
PANNS/
  audioset_tagging_cnn/pytorch/models.py
  checkpoints/Wavegram_Logmel_Cnn14_mAP=0.439.pth
  data/ESC-50/meta/esc50.csv
  data/ESC-50/audio/*.wav
```

Đặt notebook và các file `train.py`, `nr_panns.py`, `nre.py`, `rghf.py`, `two_branch_panns.py` trong **cùng một thư mục**. Chạy các cell từ trên xuống. Không chạy toàn bộ 5-fold trước khi kiểm tra fold thử.

## 1. Cài dependencies (chạy một lần, sau đó restart kernel nếu cần)
PANNs gốc có thể yêu cầu các gói bổ sung, ví dụ `librosa`, `torchlibrosa`, `h5py`. Dòng cài đặt sau phục vụ cả pipeline điển hình.

In [ ]:
%pip install -q numpy scipy soundfile pandas matplotlib tqdm torchlibrosa librosa h5py

## 2. Import & kiểm tra GPU

In [ ]:
import sys, json, csv
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

NOTEBOOK_DIR = Path.cwd().resolve()  # Mở notebook từ thư mục nrpanns_retrain
sys.path.insert(0, str(NOTEBOOK_DIR))
from train import train_fold, seed_everything, loaders_for, build_model, gradient_diagnostics
from two_branch_panns import DEFAULT_CHECKPOINT_NAME

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Cấu hình
Sửa `ROOT` thành thư mục PANNS trên máy của mày. Có thể giảm `BATCH_SIZE` xuống 2–4 nếu CUDA hết bộ nhớ. `WORKERS=0` giúp notebook ổn định trên Windows.

In [ ]:
ROOT = Path('/home/robinson/robotic/PANNS')  # <<< SỬA ĐƯỜNG DẪN NÀY
CHECKPOINT = ROOT / 'checkpoints' / DEFAULT_CHECKPOINT_NAME
NRE_CHECKPOINT = None  # Ví dụ: ROOT / 'checkpoints/nre_v3_best.pth'
OUTPUT = NOTEBOOK_DIR / 'runs' / 'nrpanns'

TEST_FOLD = 5
EPOCHS = 40
PATIENCE = 12
BATCH_SIZE = 8
WORKERS = 0
LR_BACKBONE = 1e-5
LR_NEW = 1e-4
WEIGHT_DECAY = 1e-4
CLIP = 5.0
ALPHA_INIT = 0.01
SEED = 42
USE_AMP = DEVICE.type == 'cuda'

args = SimpleNamespace(
    root=ROOT, checkpoint=CHECKPOINT, nre_checkpoint=NRE_CHECKPOINT,
    output=OUTPUT, epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE,
    workers=WORKERS, lr_backbone=LR_BACKBONE, lr_new=LR_NEW,
    weight_decay=WEIGHT_DECAY, clip=CLIP, alpha=ALPHA_INIT,
    seed=SEED, amp=USE_AMP,
)
print('Test fold:', TEST_FOLD, '| validation fold:', TEST_FOLD % 5 + 1)
print('Output:', OUTPUT)

## 4. Kiểm tra file đầu vào và chia fold
Mỗi lần chạy dùng 3 fold train, 1 fold validation và 1 fold test. Test fold **không** dùng để chọn checkpoint.

In [ ]:
required = [
    ROOT / 'audioset_tagging_cnn' / 'pytorch' / 'models.py',
    CHECKPOINT,
    ROOT / 'data' / 'ESC-50' / 'meta' / 'esc50.csv',
    ROOT / 'data' / 'ESC-50' / 'audio',
]
for p in required:
    print(('OK     ' if p.exists() else 'MISSING'), p)
if not all(p.exists() for p in required):
    raise FileNotFoundError('Thiếu file/thư mục: sửa ROOT hoặc CHECKPOINT trước khi train.')

train_loader, val_loader, test_loader, val_fold = loaders_for(
    ROOT / 'data' / 'ESC-50', TEST_FOLD, BATCH_SIZE, WORKERS, SEED + TEST_FOLD
)
print('Train/Val/Test samples:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
print('Train folds:', sorted(set(range(1,6)) - {TEST_FOLD, val_fold}))
batch = next(iter(train_loader))
print('Waveforms:', tuple(batch['waveform'].shape), 'Labels:', tuple(batch['label'].shape))

## 5. Khởi tạo mô hình và chạy thử forward
Cell này tải checkpoint pretrained, khởi tạo mô hình mới và xác minh đầu ra `[B,50]`. Không cần chạy nếu muốn chuyển thẳng sang train và thiếu VRAM.

In [ ]:
seed_everything(SEED + TEST_FOLD)
model_check = build_model(ROOT, CHECKPOINT, NRE_CHECKPOINT, ALPHA_INIT).to(DEVICE)
model_check.eval()
with torch.no_grad():
    # 1 mẫu để tránh OOM
    logits = model_check(batch['waveform'][:1].to(DEVICE))
print('Logits shape:', tuple(logits.shape))
assert logits.shape == (1, 50)
print('Trainable params:', f'{sum(p.numel() for p in model_check.parameters() if p.requires_grad):,}')
del model_check, logits
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

## 6. Train một fold (chính)
`train_fold` tự tạo model/optimizer/scheduler mới, chạy gradient check ở batch đầu, lưu `best.pt`, `history.csv`, `result.json`, sau đó đánh giá test một lần từ checkpoint tốt nhất.

In [ ]:
result = train_fold(args, TEST_FOLD, DEVICE)
print('Best epoch:', result['best_epoch'])
print('Validation accuracy:', f"{result['best_val_accuracy']:.2%}")
print('Held-out test accuracy:', f"{result['test_accuracy']:.2%}")

## 7. Biểu đồ train/validation và kết quả

In [ ]:
history_file = OUTPUT / f'fold_{TEST_FOLD}' / 'history.csv'
history = pd.read_csv(history_file)
display(history.tail())
ax = history.plot(x='epoch', y=['train_accuracy', 'val_accuracy'], marker='o', figsize=(9,4))
ax.set_ylabel('Accuracy')
ax.set_title(f'NR-PANNs - Test fold {TEST_FOLD}')
ax.grid(True, alpha=0.3)
plt.show()

ax = history.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(9,4))
ax.set_ylabel('Cross-entropy loss')
ax.grid(True, alpha=0.3)
plt.show()

print(json.dumps(json.loads((OUTPUT / f'fold_{TEST_FOLD}' / 'result.json').read_text()), indent=2))

## 8. Tùy chọn: chạy cả 5 test folds
**Chỉ chạy cell này khi muốn làm đánh giá 5-fold.** Mỗi fold tự huấn luyện lại một mô hình từ checkpoint pretrained, không tiếp tục trọng số của fold trước.

In [ ]:
RUN_ALL_FOLDS = False  # Đổi True nếu muốn train 5 folds
if RUN_ALL_FOLDS:
    results = [train_fold(args, fold, DEVICE) for fold in range(1, 6)]
    summary = {
        'folds': results,
        'mean_test_accuracy': float(np.mean([r['test_accuracy'] for r in results])),
        'std_test_accuracy': float(np.std([r['test_accuracy'] for r in results])),
    }
    OUTPUT.mkdir(parents=True, exist_ok=True)
    (OUTPUT / 'summary.json').write_text(json.dumps(summary, indent=2))
    display(pd.DataFrame(results))
    print('Mean test accuracy:', f"{summary['mean_test_accuracy']:.2%}")
else:
    print('Bỏ qua 5-fold. Đặt RUN_ALL_FOLDS=True để chạy.')

## 9. Tải lại checkpoint tốt nhất để inference
Cell này lấy đúng checkpoint của fold đang chọn. Khi dự đoán waveform phải mono, 32 kHz, 160000 samples.

In [ ]:
best_path = OUTPUT / f'fold_{TEST_FOLD}' / 'best.pt'
if best_path.exists():
    inference_model = build_model(ROOT, CHECKPOINT, NRE_CHECKPOINT, ALPHA_INIT).to(DEVICE)
    state = torch.load(best_path, map_location=DEVICE, weights_only=True)
    inference_model.load_state_dict(state['model'])
    inference_model.eval()
    sample = next(iter(test_loader))
    with torch.inference_mode():
        pred = inference_model(sample['waveform'][:1].to(DEVICE)).argmax(1).item()
    true_id = int(sample['label'][0])
    print('Predicted class index:', pred, '| True class index:', true_id)
    del inference_model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('Chưa có best.pt. Chạy cell train một fold trước.')

### Ghi chú
- File `best.pt` chứa trọng số **NR-PANNs sau fine-tuning**, không chỉ classifier.
- Nếu muốn train NRE-V3 riêng trên clean/noisy trước khi fine-tune NR-PANNs, cần huấn luyện NRE bằng `nre_experiment.py` rồi đặt `NRE_CHECKPOINT` trỏ tới checkpoint đó.
- Đây là pipeline **train lại task ESC-50 từ AudioSet-pretrained PANNs**. Nếu muốn train PANNs từ random weights thì phải sửa `TwoBranchPANNs` để bỏ load checkpoint, đồng thời thay đổi đáng kể cấu hình train.
- Chưa thể chạy train thực tế trong môi trường hiện tại nếu chưa có thư mục dữ liệu, mã nguồn và checkpoint pretrained PANNs gốc.